# Echelon Chess Engine - Colab Training (C++ Optimized)

This notebook trains the Echelon chess engine using high-performance C++ backend for AlphaZero-style self-play.

**Before running:**
1. Go to Runtime > Change runtime type > GPU (T4)
2. Upload or clone the echelon repository

In [ ]:
# Clone repository (or upload files)
!git clone https://github.com/falloficarus22/echelon.git
%cd echelon

In [ ]:
# Install dependencies
!pip install pybind11 -q

In [ ]:
# Compile high-performance C++ backend IN COLAB
# This ensures compatibility with Colab's environment
import os
import subprocess

os.chdir('cpp')

# Get Python config for current environment
ext_suffix = subprocess.check_output(['python3-config', '--extension-suffix']).decode().strip()
print(f"Extension suffix: {ext_suffix}")

# Compile with proper flags
!g++ -O3 -Wall -shared -std=c++17 -fPIC \
    $(python3 -m pybind11 --includes) \
    attacks.cpp magic.cpp board.cpp mcts.cpp echelon_cpp_wrapper.cpp \
    -o echelon_cpp{ext_suffix}

os.chdir('..')
print("\n✓ C++ backend compiled successfully!")

In [ ]:
# Verify the module loads correctly
import sys
sys.path.insert(0, './cpp')

try:
    import echelon_cpp
    echelon_cpp.init()
    print("✓ C++ module loaded successfully!")
    print("✓ Attack tables initialized!")
except Exception as e:
    print(f"✗ Error loading module: {e}")
    raise

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Training

**Recommended settings for T4 (15GB VRAM):**
- Using `train_cpp.py` for maximum speed (60x faster self-play)
- Higher MCTS simulations are now feasible thanks to C++

**Note:** The training script will automatically use GPU if available.

In [ ]:
# Start training using C++ backend
# This will run 10 iterations with 5 games each
!python train_cpp.py

## Save Results to Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy checkpoints to Drive
import shutil
import os

drive_path = '/content/drive/MyDrive/echelon_checkpoints'
os.makedirs(drive_path, exist_ok=True)

# Copy all .pt files
!cp *.pt {drive_path}/ 2>/dev/null || echo 'No checkpoints found yet'
print(f"Checkpoints saved to {drive_path}")